# S04 · Git as save points you can walk back to

Git gives your project **save points**, like the save points in a video game. In
this notebook we turn a plain folder into a Git project, make two save points, look
back at an older one, and tell Git what to ignore. Everything happens inside a
throwaway folder that cleans itself up, so nothing on your computer is touched.

At the end there is an **optional stretch section** on branches, merging and merge
conflicts. We do not cover those in class — they are there for when you are ready.

**New here? Read this once.**

- Never opened a terminal in your life? Perfect, this notebook is for you. Just
  press the play button on each cell, top to bottom, and read the one-line note
  above it. You will see every Git command run for real.
- The exact same commands, in a list you can copy for the lab, are in
  `../data/README.md`.
- Curious why a history of save points is a "graph"? Open the primer
  `primers/graphs_and_dags.md`. It is optional and all pictures.
- Already comfortable with Git? Skip to the cell marked **Stretch (optional)** near
  the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

Git is a program that comes ready on **Google Colab** and that you installed on
your **own machine** for this course. It is not a Python library, so there is
nothing to `pip install`. The next cell just checks Git is there.

In [ ]:
# Check that Git is available. This prints the version and confirms we are good.
!git --version

## Make a throwaway folder to play in

We do not want to practise on anything important, so we make a fresh temporary
folder and step into it. `tempfile` gives us a folder the system will happily
throw away later. We remember where we started so we can come back and delete it
at the end.

In [ ]:
import os          # to move between folders and read paths
import tempfile     # to make a safe throwaway folder
import shutil       # to delete that folder cleanly at the end

# Remember where we started, so we can return here later.
starting_folder = os.getcwd()

# Make a brand-new empty folder and step into it.
practice_folder = tempfile.mkdtemp(prefix="git-practice-")
os.chdir(practice_folder)

print("We are now working inside a throwaway folder:")
print(practice_folder)

## `git init` — start watching this folder

`git init` tells Git to start watching the current folder and to keep its history.
It creates a hidden `.git` folder where every save point will be stored. We pass
`-b main` so the first line of history is called `main`, the usual name.

In [ ]:
# Turn this plain folder into a Git project whose main line is called "main".
!git init -b main

## Tell Git who you are

Every save point records who made it, so Git needs a name and an email. Here we set
them just for this practice folder. In the lab you set them once for your whole
computer instead, by adding `--global` to these two commands.

In [ ]:
# Set a name and email for this practice folder only (no --global here).
!git config user.name  "A Learner"
!git config user.email "learner@example.com"
print("Identity set for this folder.")

## Create a first file

A save point needs something to save. We write a small notes file for our pretend
project. In the real lab this would be your README and your Session 1–3 notebooks.

In [ ]:
# Write a small text file into our project folder.
with open("analysis_notes.txt", "w") as notes_file:
    notes_file.write("Project: monsoon rainfall vs crop yield\n")
    notes_file.write("Day 1: loaded the data, it has 5 years of records.\n")

print("Wrote analysis_notes.txt")

## `git status` — what has changed?

`git status` is the command you will run most. It shows what is new or changed and
what is ready for the next save point. Right now it should report that
`analysis_notes.txt` is new and not yet saved.

In [ ]:
# Ask Git what it currently sees in the folder.
!git status

## `git add` — put the file on the tray

Before a save point, you choose exactly what goes into it by putting files on a
"tray" (Git calls this staging). `git add` places our file on the tray. This lets a
save point be neat and focused instead of grabbing everything at once.

In [ ]:
# Put analysis_notes.txt on the tray for the next save point.
!git add analysis_notes.txt

# Look again: the file is now "staged", i.e. on the tray, ready to be saved.
!git status

## `git commit` — freeze the tray as a save point

`git commit` freezes whatever is on the tray into a permanent save point. The
`-m` part is the message: a short note to future-you saying what this save point
is. Write it like a helpful label, not "stuff".

In [ ]:
# Make our first save point, with a clear message.
!git commit -m "Start the analysis notes"


## `git log` — see your save points

`git log --oneline` lists your save points, newest first, one per line. Each has a
short code (its id) and the message you wrote. Right now there is exactly one.

In [ ]:
# Show the history: our save points so far.
!git log --oneline

## Change the file and make a second save point

Real work is many small save points. We add a line to the notes, then run the same
three steps again: check with `status`, put it on the tray with `add`, freeze it
with `commit`. Notice how the rhythm repeats.

In [ ]:
# Add a new finding to the notes file.
with open("analysis_notes.txt", "a") as notes_file:
    notes_file.write("Day 2: rainfall and yield rise together - promising!\n")

# The same rhythm: see what changed, tray it, save it.
!git add analysis_notes.txt
!git commit -m "Add the day-2 finding"


## Two save points now

Run the log again. There are two save points, newest at the top. This little list,
each save point pointing back to the one before it, is your project's history.

In [ ]:
# Show the history again: two save points now.
!git log --oneline

## The time machine — look back, and bring a file back

This is the payoff. `HEAD~1` means "one save point before the newest". So
`git show HEAD~1:analysis_notes.txt` prints the notes file exactly as it was at our
first save point, before we added the day-2 line. Looking back is never lost to you.

In [ ]:
# Show the notes file as it was at the previous save point.
!git show HEAD~1:analysis_notes.txt

### Recover a file from an earlier save point

Looking is nice, but the real fear is **losing** a file — you delete the block that
mattered, or the whole file, right before a deadline. This is the one-line cure.

`git restore --source=<save point> <file>` copies that file out of an earlier save
point and drops it back into your project. We wreck the notes file on purpose, then
bring it straight back.

In [ ]:
# Simulate the horror: wipe the notes file entirely.
import os
os.remove("analysis_notes.txt")
!git status --short          # Git sees it as deleted

# Bring it back exactly as it was at the LATEST save point. One line.
!git restore --source=HEAD analysis_notes.txt
print("--- analysis_notes.txt is back ---")
print(open("analysis_notes.txt").read())

And you can reach any save point, not just the latest. `HEAD~1` is one save point
back — the version before we added the day-2 line. Restoring from it puts that older
version in your working file; restoring from `HEAD` again returns to the newest, so
nothing is lost either way.

In [ ]:
# Pull the OLDER version (day-1 only) back into the working file.
!git restore --source=HEAD~1 analysis_notes.txt
print("--- restored from one save point ago ---")
print(open("analysis_notes.txt").read())

# Change our mind: return to the latest version, so we carry on cleanly.
!git restore --source=HEAD analysis_notes.txt
print("--- back to the latest save point ---")
print(open("analysis_notes.txt").read())

## Keep junk out with a `.gitignore`

Some things should never go into a save point: your virtual environment, huge data
files, and above all secrets like passwords or API keys (a committed secret stays
in the history forever). A `.gitignore` file lists patterns Git should ignore. We
write one, then check that Git now leaves those files alone.

In [ ]:
# List things Git should never save.
with open(".gitignore", "w") as ignore_file:
    ignore_file.write(".venv/\n")
    ignore_file.write("__pycache__/\n")
    ignore_file.write("*.csv\n")
    ignore_file.write(".env\n")

# Pretend we created a virtual-environment folder full of junk.
os.makedirs(".venv", exist_ok=True)
with open(".venv/some_library_file", "w") as junk_file:
    junk_file.write("machine-specific junk\n")

# git status should show .gitignore as new, but NOT the .venv folder.
!git status

---

## Stretch (optional) — branches, merging, and your first conflict

Everything above is the whole of Session 4. **You can stop here.**

This section is for when you are curious, or working on a project with someone
else. It is the part of Git that sounds frightening and turns out not to be. Work
through it at your own pace; nothing later in the course depends on it.

### A branch is a parallel line of history

A **branch** is a parallel line of save points. You make one to try a wild idea
without touching your good work on `main`. If the idea flops, you delete the branch
and `main` never knew about it.

First we save the `.gitignore` we just wrote, so we start this section from a clean
slate — the same `add` / `commit` rhythm as before.

In [ ]:
# Save the .gitignore from the previous step, so nothing is left hanging.
!git add .gitignore
!git commit -m "Ignore junk and data files"

# Create a new branch and move onto it, in one command.
!git switch -c try-an-idea

# List the branches; a star marks the one we are on.
!git branch

### Work on the branch, then bring it back with a merge

We add a finding while we are on `try-an-idea`, then switch back to `main` and look
at the file. The finding is **absent** — same folder, different line of history.
That is the moment branches click.

`git merge` then brings the branch's work into `main`.

In [ ]:
# We are on try-an-idea. Add a Day 3 finding here.
with open("analysis_notes.txt", "a") as notes_file:
    notes_file.write("Day 3: a log scale fits the curve better.\n")

!git add analysis_notes.txt
!git commit -m "Try a log scale"

# Go back to the main line and read the file there.
!git switch main
print("\n--- analysis_notes.txt on main ---")
print(open("analysis_notes.txt").read())
print("Notice: no Day 3 line. The work is safe on the branch, not here.")

In [ ]:
# Now bring the branch's work into main.
!git merge try-an-idea

print("\n--- analysis_notes.txt on main, after the merge ---")
print(open("analysis_notes.txt").read())

# The branch has served its purpose; tidy it away.
!git branch -d try-an-idea

### A merge conflict, on purpose

That merge was easy: `main` had not moved, so Git just slid the label forward (it
calls this a *fast-forward*).

A **conflict** happens when two lines of history change the *same place* in a file
differently. Git will not guess which is right, so it stops and asks you. It is not
an error and it is not a bug — it is Git refusing to silently throw away someone's
work.

Let us cause one deliberately. Two people reach opposite conclusions about the same
result on Day 4.

In [ ]:
# Ravi's view, on a branch.
!git switch -c careful-reading
with open("analysis_notes.txt", "a") as notes_file:
    notes_file.write("Day 4: correlation is 0.71 - strong. Ship it.\n")
!git add analysis_notes.txt
!git commit -m "Day 4: strong correlation"

# Asha's view, on main, about the SAME last line of the file.
!git switch main
with open("analysis_notes.txt", "a") as notes_file:
    notes_file.write("Day 4: correlation is 0.71, but 5 years is a small sample.\n")
!git add analysis_notes.txt
!git commit -m "Day 4: correlation, with a caveat"

# The two lines of history now disagree. Merge them and watch.
!git merge careful-reading

### Read the conflict, then decide

Git has stopped and written **both** versions into the file, wrapped in markers:

```
<<<<<<< HEAD
   ... the version on the branch you are ON (main)
=======
   ... the version from the branch you are MERGING IN
>>>>>>> careful-reading
```

Nothing is broken. Git is showing you both and waiting. Print the file and look.

In [ ]:
# Look at what Git actually put in the file.
print(open("analysis_notes.txt").read())

### Resolve it

Resolving is not a special Git command. You **edit the file** until it says what you
want, delete all three marker lines, then `add` and `commit` exactly as always.

In your editor you would do this by hand. Here we write the resolved file from
Python so the notebook can run start to finish — but the act is the same: keep the
line you want, remove the markers.

In [ ]:
# Keep Asha's more careful wording, and drop the markers entirely.
resolved = (
    "Project: monsoon rainfall vs crop yield\n"
    "Day 1: loaded the data, it has 5 years of records.\n"
    "Day 2: rainfall and yield rise together - promising!\n"
    "Day 3: a log scale fits the curve better.\n"
    "Day 4: correlation is 0.71, but 5 years is a small sample.\n"
)
with open("analysis_notes.txt", "w") as notes_file:
    notes_file.write(resolved)

# The same old rhythm: stage the fix, then commit to complete the merge.
!git add analysis_notes.txt
!git commit -m "Merge careful-reading: keep the caveat on Day 4"

print("\n--- resolved ---")
print(open("analysis_notes.txt").read())

### That is the whole of it

You caused a conflict, read it, and fixed it in three commands. The first one in a
real project still gives everyone a jolt, but you have now seen that it is just Git
being honest about two humans disagreeing.

The graph below is worth a look: the two lines of history split apart and come back
together at a save point with **two parents**. That is the merge.

### The history as a graph — the DAG

`git log --graph` draws the shape of the history. Because we branched and merged
above, it is no longer a straight chain: it splits where we started a branch and
fans back together where we merged.

That picture is the **directed acyclic graph** the session's optional box talks
about — dots (save points) joined by one-way arrows, with no way to ever loop back
to where you started. The merge is the node with *two* arrows leaving it, one to
each parent. The primer `primers/graphs_and_dags.md` has the full picture.

In [ ]:
# Draw the history as a small graph (one line each, oldest at the bottom).
!git log --oneline --graph --all

## Sending it to GitHub (this is the lab)

Everything so far lived only on this computer. To make it **safe from a dying
laptop** and **shareable with a link**, you push it to GitHub. We do not run these
here because they need your own GitHub account and a live internet connection, but
this is exactly what you will type in the lab, after creating an empty repo on the
GitHub website:

```bash
# Point your project at the empty repo you made on github.com
git remote add origin https://github.com/<your-username>/my-first-repo.git

# Send every save point up to GitHub
git branch -M main
git push -u origin main
```

After that, your work lives online. Anyone you share the link with can see it, and
a recruiter opening it sees a real project with a clear history.

## Clean up

We step back to where we started and delete the throwaway folder, so nothing is
left behind. In your real project you would of course keep the folder.

In [ ]:
# Return to the original folder and delete the practice one.
os.chdir(starting_folder)
shutil.rmtree(practice_folder)
print("Cleaned up. The practice folder is gone; your machine is untouched.")

## What you just did

You turned a folder into a Git project, made two save points, looked back at an
earlier version, and told Git what to ignore. That is the whole everyday rhythm:
**edit, `add`, `commit`**, again and again, with `push` to send it to GitHub.

In the lab you do exactly this for real, then push your Session 1–3 labs to GitHub
as your first public repository. The command cheat-sheet is in `../data/README.md`
and a README + `requirements.txt` you can copy are in `../repo-template/`.

Next notebook: `02_reading_tracebacks.ipynb`, on staying calm when code breaks.